- Generates and saves Spectral Plots for Kasios claimed Rose-crested Blue Pipit audio files
- Saves spectral plot stats

In [ ]:
import pandas as pd
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# load kasios metadata and preview
kasios_bird_locations = pd.read_csv("/content/drive/MyDrive/visual_analytics_project/Test Birds Location.csv")
kasios_bird_locations.head()

,ID,X,Y
0,1,140,119
1,2,63,153
2,3,70,136
3,4,78,150
4,5,60,90


In [ ]:
print(f"Total recordings: {len(kasios_bird_locations)}")

Total recordings: 15


In [ ]:
# access audio folder
kasios_birds = "/content/drive/MyDrive/visual_analytics_project/kasios_birds"

In [ ]:
# function to plot spectral data for a given audio file path
def plot_spectral(audio_path, title="Spectral Plot", save_path=None):
    # load audio file
    y, sr = librosa.load(audio_path)

    # compute FFT: magnitude spectrum
    fft = np.fft.fft(y)
    magnitude = np.abs(fft)
    freq = np.fft.fftfreq(len(magnitude), 1/sr)

    # keep only the positive half of the spectrum
    left_freq = freq[:len(freq)//2]
    left_magnitude = magnitude[:len(magnitude)//2]

    # plot
    plt.figure(figsize=(10, 4))
    plt.plot(left_freq, left_magnitude)
    plt.title(title)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.grid(True)
    plt.tight_layout()

    # save the plot to a file
    if save_path:
        plt.savefig(save_path)
        plt.close()  # close to avoid open figures
    else:
        plt.show()

In [ ]:
# specify output directory
output_dir = "/content/drive/MyDrive/visual_analytics_project/spectral_plots/kasios_sounds"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# list to save spectral data
spectral_data = []

In [ ]:
# process all sounds and generate spectral plots
for _, row in kasios_bird_locations.iterrows():
    # extract file ID
    filename = row['ID']
    file_id = filename

    # construct file path
    file_path = os.path.join(kasios_birds, f"{file_id}.mp3")

    # define title and save path for plot
    title = f"Spectral Plot - Kasios {file_id}"
    save_path = os.path.join(output_dir, f"{file_id}_spectral_plot.png")

    # plot and save
    plot_spectral(file_path, title, save_path)

    # add statistics to data list and extract
    y, sr = librosa.load(file_path)
    fft = np.fft.fft(y)
    magnitude = np.abs(fft)
    freq = np.fft.fftfreq(len(magnitude), 1/sr)

    left_freq = freq[:len(freq)//2]
    left_magnitude = magnitude[:len(magnitude)//2]

    peak_freq = left_freq[np.argmax(left_magnitude)]
    mean_freq = np.mean(left_freq)
    total_magnitude = np.sum(left_magnitude)
    entropy = -np.sum((left_magnitude / total_magnitude) * np.log2(left_magnitude / total_magnitude + 1e-10))

    spectral_data.append({
        'File ID': file_id,
        'Peak Frequency': peak_freq,
        'Mean Frequency': mean_freq,
        'Total Magnitude': total_magnitude,
        'Spectral Entropy': entropy
    })

In [ ]:
# save spectral stats
pd.DataFrame(spectral_data).to_csv("kasios_spectral_statistics.csv", index=False)